In [ ]:
import pandas as pd
import re

def ground_truths_escenario2(log_text):
    log = str(log_text).upper()
    
    # Regla 1 estricta: cat /etc/shadow en HEX DEBE ser ALTO
    if '636174202F6574632F736861646F77' in log:
        return ['ALTO']
        
    # Regla 2: Otros intentos de escalada o auth fallida
    elif 'RES=FAILED' in log and ('TYPE=USER_CMD' in log or 'TYPE=USER_AUTH' in log):
        return ['ALTO', 'MEDIO']
        
    # Manejo de errores de parseo del LLM
    elif 'ERROR_PARSE' in log or 'FALLO_LLM' in log:
        return ['ERROR']
        
    # Por defecto
    return ['INFO', 'BAJO', 'INFORMACIÓN']

def evaluar_por_orden(path_completo, path_evaluar, funcion_heuristica):
    """
    Evalúa la precisión comparando fila por fila basándose en el orden estricto de los CSV.
    """
    df_completo = pd.read_csv(path_completo)
    df_evaluar = pd.read_csv(path_evaluar)
    
    # Verificación de integridad secuencial
    if len(df_completo) != len(df_evaluar):
        print(f"¡ALERTA!: Archivos desfasados. Completo: {len(df_completo)} | A evaluar: {len(df_evaluar)}")
        return
        
    # Construir la lista de la verdad absoluta
    verdad_en_orden = df_completo['Log'].apply(funcion_heuristica).tolist()
    
    aciertos = 0
    total_validos = 0
    
    # Comparar fila por fila
    for i in range(len(df_evaluar)):
        prediccion_llm = str(df_evaluar.loc[i, 'Riesgo']).upper().strip()
        etiquetas_reales = verdad_en_orden[i]
        
        if 'ERROR' not in etiquetas_reales:
            total_validos += 1
            if prediccion_llm in etiquetas_reales:
                aciertos += 1
                
    precision = (aciertos / total_validos) * 100 if total_validos > 0 else 0
    print(f"Precisión para {path_evaluar}: {precision:.2f}% ({aciertos}/{total_validos})")
    return precision


In [ ]:
# --- EJECUCIÓN ---
# Definir la ruta del archivo que tiene la verdad (RAW Completo)
ruta_verdad = '../results/prompt1/escenario2_resultados_raw_completo_phi3mini.csv'

# Evaluar el RAW Completo contra sí mismo para la nota base
evaluar_por_orden(ruta_verdad, ruta_verdad, ground_truths_escenario2)

# Evaluar los otros formatos
evaluar_por_orden(ruta_verdad, '../results/prompt1/escenario2_resultados_raw_reducido_phi3mini.csv', ground_truths_escenario2)
evaluar_por_orden(ruta_verdad, '../results/prompt1/escenario2_resultados_json_reducido_phi3mini.csv', ground_truths_escenario2)
evaluar_por_orden(ruta_verdad, '../results/prompt2/escenario2_resultados_raw_reducido_phi3mini.csv', ground_truths_escenario2)
evaluar_por_orden(ruta_verdad, '../results/prompt3/escenario2_resultados_raw_reducido_phi3mini.csv', ground_truths_escenario2)

Precisión para ../results/prompt1/escenario2_resultados_raw_completo_phi3mini.csv: 85.71% (6/7)
Precisión para ../results/prompt1/escenario2_resultados_raw_reducido_phi3mini.csv: 28.57% (2/7)
Precisión para ../results/prompt1/escenario2_resultados_json_reducido_phi3mini.csv: 85.71% (6/7)
Precisión para ../results/prompt2/escenario2_resultados_raw_reducido_phi3mini.csv: 100.00% (7/7)
Precisión para ../results/prompt3/escenario2_resultados_raw_reducido_phi3mini.csv: 57.14% (4/7)


57.14285714285714